In [21]:
 # ! If crashes
#python -m ipykernel install --user --name torch_stable --display-name "Python (torch_stable)"
#import os
#os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
#os.environ["OMP_NUM_THREADS"] = "1"

In [1]:
import torch as tp
import numpy as np 
import pandas as pd 
print(np.__version__)
print(pd.__version__)
print(tp.__version__)

1.26.4
2.2.0
2.5.1


In [3]:
import pandas as pd 

data=pd.read_csv('100_Unique_QA_Dataset.csv')
data.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
 # TODO: Tokenizing the data

def tokenize(text):
    text=text.lower()
    text=text.replace('?','')
    text=text.replace("'","")
    return text.split()

In [5]:
# Vocab 
vocab={'<UNK>':0}

def build_vocab(row):
    tokenized_question=tokenize(row['question'])
    tokenized_ans=tokenize(row['answer'])

    merged_tokens=tokenized_question + tokenized_ans

    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)



In [6]:
data.apply(build_vocab,axis=1)


0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [7]:
 # Convert words to numerical indices 

def text_to_indices(text,vocab):
    indexed_text=[]

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text
    
 # TODO: text_to_indices('Captial of india?',vocab) # Output [0, 5, 73]

In [8]:
import sys
print(sys.version)


3.11.9 | packaged by Anaconda, Inc. | (main, Apr 19 2024, 16:40:41) [MSC v.1916 64 bit (AMD64)]


In [9]:
import torch as tp
print(tp.__version__)

2.5.1


In [10]:
import torch as tp
from torch.utils.data import DataLoader,Dataset

class QADataset(Dataset):

    def __init__(self,data,vocab):
        self.data=data
        self.vocab=vocab
        
    def __len__(self):
        return self.data.shape[0]
    def __getitem__(self,index):
        num_quest=text_to_indices(self.data.iloc[index]['question'],self.vocab)
        num_ans=text_to_indices(self.data.iloc[index]['answer'],self.vocab)

        return tp.tensor(num_quest),tp.tensor(num_ans)


In [11]:
dataset=QADataset(data,vocab)


In [12]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [13]:
for question , answer in dataloader:
    print(question,answer)

tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[ 10,  96,   3, 104, 239]]) tensor([[240]])
tensor([[ 42, 137,   2,  62,  39,   3, 322, 323]]) tensor([[6]])
tensor([[ 10,  11, 189, 158, 190]]) tensor([[191]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([[170]])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([[155]])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([[68]])
tensor([[  1,   2,   3, 146, 147,  19, 148]]) tensor([[149]])
tensor([[ 1,  2,  3, 92, 93, 94]]) tensor([[95]])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([[74]])
tensor([[ 10,  75,   3, 296,  19, 297]]) tensor([[298]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[  1,   2,   3,  92, 137,  19,   3, 

In [14]:
import torch.nn as nn 
class SimpleRNN(nn.Module):
    pass
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50) #! This is embedding Layer
        self.rnn=nn.RNN(50,90,batch_first=True)
        self.fc=nn.Linear(90,vocab_size)

    def forward(self,question):
        embedded_quest=self.embedding(question)
        hidden,final=self.rnn(embedded_quest)
        output=self.fc(final.squeeze(0))

        return output


In [15]:
learn_rate=0.001
epochs=20


In [16]:
model=SimpleRNN(len(vocab))

In [17]:
criterion=nn.CrossEntropyLoss()
optimizer=tp.optim.Adam(model.parameters(),lr=learn_rate)

In [18]:
# Training Loop
for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        #! Forward pass
        output=model(question)
        #! loss
        loss=criterion(output,answer[0])
        #! gradient
        loss.backward()
        #! update gradients
        optimizer.step()
        total_loss=total_loss+loss.item()
    print(f'Epochs: {epoch+1}, Loss: {total_loss}')


Epochs: 1, Loss: 526.4265632629395
Epochs: 2, Loss: 437.9235932826996
Epochs: 3, Loss: 342.66342878341675
Epochs: 4, Loss: 268.7329275608063
Epochs: 5, Loss: 203.39401423931122
Epochs: 6, Loss: 145.66606265306473
Epochs: 7, Loss: 101.87075248360634
Epochs: 8, Loss: 69.14996910095215
Epochs: 9, Loss: 48.95438924431801
Epochs: 10, Loss: 35.56735406816006
Epochs: 11, Loss: 26.44096054881811
Epochs: 12, Loss: 20.048788867890835
Epochs: 13, Loss: 15.88262390345335
Epochs: 14, Loss: 12.715777207165956
Epochs: 15, Loss: 10.391587685793638
Epochs: 16, Loss: 8.586037937551737
Epochs: 17, Loss: 7.3044044859707355
Epochs: 18, Loss: 6.2613823637366295
Epochs: 19, Loss: 5.456450438126922
Epochs: 20, Loss: 4.757969107478857


In [19]:
 #? Prediction
def predict(model,question,threshold=0.3):

    #! convert Q to numbers
    num_quest=text_to_indices(question,vocab)
    #! tensor
    quest_tensor=tp.tensor(num_quest).unsqueeze(0)
    #! send to model
    output=model(quest_tensor)
    
    #print(output.shape)
    #!convert logits to probab
    probab=tp.nn.functional.softmax(output,dim=1)
    
    #! find index of max probab
    value,index=tp.max(probab,dim=1)

    if value<threshold:
        print("I Dont Know")

    print(list(vocab.keys())[index])


In [20]:
predict(model,'what is the capital of india')

delhi
